In [1]:
# pip install -r requirements.txt

In [2]:
import numpy as np 
import pandas as pd 
import torch
import plotly.express as px
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio

In [3]:
pio.renderers.default = 'notebook_connected' 

# Based on the article A Survey of Quantization Methods for Efficient Neural Network Inference by Gholami et al. [ResearchGate](https://www.researchgate.net/publication/357784540_A_Survey_of_Quantization_Methods_for_Efficient_Neural_Network_Inference) & [arXiv](https://arxiv.org/abs/2103.13630)

## Quantization -- Uniform

$Q(r)=Int(r/S)-Z$ where:
* $Q$ is the quantization operator
* $r$ is real valued input (activation or weights)
* $S$ is a real valued scaling factor
* $Z$ is an integer zero point

In [4]:
def quantization_operator(r,S,Z):
    r_quantized=np.zeros(r.shape)
    if len(r.shape) == 1:
        for i in range(r.shape[0]):
            r_quantized[i]= int(r[i]/S)-Z
    else:
        for i in range(r.shape[0]):
            for j in range(r.shape[1]):
               r_quantized[i,j]= int(r[i,j]/S)-Z 
    return r_quantized

## Dequantization

$\tilde{r}=S(Q(r)+Z$ where:
* $Q$ is the quantization operator
* $r$ is real valued input (activation or weights)
* $S$ is a real valued scaling factor
* $Z$ is an integer zero point

In [5]:
def dequantization_operator(r_quantized,S,Z):
    r_dequantized=np.zeros(r_quantized.shape)
    if len(r_quantized.shape) == 1:
        for i in range(r_quantized.shape[0]):
            r_dequantized[i,j]= S*(r_quantized[i,j] + Z)
    else:
        for i in range(r_quantized.shape[0]):
            for j in range(r_quantized.shape[1]):
               r_dequantized[i,j]= S*(r_quantized[i,j] + Z)
    return r_dequantized

# Experiments with quantization and dequanization

In [6]:
tensor=np.array([[-2.0,-1.0, 0.0, 1.0, 2.0]])

In [7]:
tensor_qua=quantization_operator(tensor,0.1,0)
tensor_qua

array([[-20., -10.,   0.,  10.,  20.]])

In [8]:
tensor_dequa=dequantization_operator(tensor_qua,0.1,0)
tensor_dequa

array([[-2., -1.,  0.,  1.,  2.]])

In [9]:
qua_error=tensor - tensor_dequa
qua_error

array([[0., 0., 0., 0., 0.]])

# Here we desided the scale point and zero point.
# In the theory authors mentioned that the Scaling factor ans Zeropoint can be defined in 2 ways. So, there are exest Symmetric and Asymmetric Quantization

## Symmetric and Asymmetric Quantization

In this case we need to define Scaling factor $S$.

$S = \dfrac{\beta - \alpha}{2^b -1}$ where $[\alpha,\beta]$ denoted the clipping range, $b$ is a quantization bit weight.


The porocess of choosing a clipping range is calling the **calibration**.

## Asymmetric Quantization

$\alpha = r_{min}$ and $\beta=r_{max}$ where $-\alpha \neq \beta$

## Symmetric Quantization


$\alpha = -\beta$ and a popular choice: $-\alpha = \beta = max(|r_{max}|, |r_{min}|)$

## Experiments for INT8

In [10]:
tensor_arr=np.array([[-2.0,-1.0, 0.0, 1.0, 2.0,3.0]])
tensor_matrix=np.matrix([[-2.0,-1.0, 0.0, 1.0, 2.0,3.0], [-2.5,-1.5, 0.0, 1.5, 2.5,3.5],[-2.5,-1.5, 0.5, 1.5, 2.5,3.5]])

In [11]:
def scale_symmetric(tensor):
    r_min=np.min(tensor)
    r_max=np.max(tensor)
    n=8
    return (2*np.maximum(np.abs(r_min),np.abs(r_max)))/(2**(n)-1)

In [12]:
def scale_asymmetric(tensor):
    r_min=np.min(tensor)
    r_max=np.max(tensor)
    n=8
    return (r_max-r_min)/(2**(n)-1)

In [13]:
def zero_point_asymmetric(tensor):
    r_min=np.min(tensor)
    r_max=np.max(tensor)
    n=8
    scale=scale_asymmetric(tensor)
    return round((int(r_min/scale)+int(r_max/scale)+1)/2)

In [14]:
S_symm_array=scale_symmetric(tensor_arr)
S_symm_array

np.float64(0.023529411764705882)

In [15]:
S_symm_matrix=scale_symmetric(tensor_matrix)
S_symm_matrix

np.float64(0.027450980392156862)

In [16]:
S_asym_array=scale_asymmetric(tensor_arr)
S_asym_array

np.float64(0.0196078431372549)

In [17]:
S_asym_matrix=scale_asymmetric(tensor_matrix)
S_asym_matrix

np.float64(0.023529411764705882)

In [18]:
zp_array=zero_point_asymmetric(tensor_arr)
zp_array

26

In [19]:
zp_matrix=zero_point_asymmetric(tensor_matrix)
zp_matrix

22

In [20]:
qua_array_sym=quantization_operator(tensor_arr,S_symm_array,0)
qua_array_asym=quantization_operator(tensor_arr,S_asym_array,zp_array)
qua_array_sym,qua_array_asym

(array([[-85., -42.,   0.,  42.,  85., 127.]]),
 array([[-128.,  -77.,  -26.,   25.,   76.,  127.]]))

In [21]:
dequa_array_sym=dequantization_operator(qua_array_sym,S_symm_array,0)
dequa_array_asym=dequantization_operator(qua_array_asym,S_asym_array,zp_array)
dequa_array_sym,dequa_array_asym

(array([[-2.        , -0.98823529,  0.        ,  0.98823529,  2.        ,
          2.98823529]]),
 array([[-2., -1.,  0.,  1.,  2.,  3.]]))

In [22]:
qua_error_array_sym= tensor_arr - dequa_array_sym 
qua_error_array_asym= tensor_arr - dequa_array_asym 
qua_error_array_sym,qua_error_array_asym

(array([[ 0.        , -0.01176471,  0.        ,  0.01176471,  0.        ,
          0.01176471]]),
 array([[0., 0., 0., 0., 0., 0.]]))

In [23]:
qua_matrix_symmetric=quantization_operator(tensor_matrix,S_symm_matrix,0)
qua_matrix_asymmetric=quantization_operator(tensor_matrix,S_asym_matrix,zp_matrix)
qua_matrix_symmetric, qua_matrix_asymmetric

(array([[-72., -36.,   0.,  36.,  72., 109.],
        [-91., -54.,   0.,  54.,  91., 127.],
        [-91., -54.,  18.,  54.,  91., 127.]]),
 array([[-107.,  -64.,  -22.,   20.,   63.,  105.],
        [-128.,  -85.,  -22.,   41.,   84.,  126.],
        [-128.,  -85.,   -1.,   41.,   84.,  126.]]))

In [24]:
dequa_matrix_sym=dequantization_operator(qua_matrix_symmetric,S_symm_matrix,0)
dequa_matrix_asym=dequantization_operator(qua_matrix_asymmetric,S_asym_matrix,zp_matrix)
dequa_matrix_sym,dequa_matrix_asym

(array([[-1.97647059, -0.98823529,  0.        ,  0.98823529,  1.97647059,
          2.99215686],
        [-2.49803922, -1.48235294,  0.        ,  1.48235294,  2.49803922,
          3.48627451],
        [-2.49803922, -1.48235294,  0.49411765,  1.48235294,  2.49803922,
          3.48627451]]),
 array([[-2.        , -0.98823529,  0.        ,  0.98823529,  2.        ,
          2.98823529],
        [-2.49411765, -1.48235294,  0.        ,  1.48235294,  2.49411765,
          3.48235294],
        [-2.49411765, -1.48235294,  0.49411765,  1.48235294,  2.49411765,
          3.48235294]]))

In [25]:
qua_error_matrix_sym= tensor_matrix - dequa_matrix_sym 
qua_error_matrix_asym= tensor_matrix - dequa_matrix_asym 
qua_error_matrix_sym,qua_error_matrix_asym

(matrix([[-0.02352941, -0.01176471,  0.        ,  0.01176471,  0.02352941,
           0.00784314],
         [-0.00196078, -0.01764706,  0.        ,  0.01764706,  0.00196078,
           0.01372549],
         [-0.00196078, -0.01764706,  0.00588235,  0.01764706,  0.00196078,
           0.01372549]]),
 matrix([[ 0.        , -0.01176471,  0.        ,  0.01176471,  0.        ,
           0.01176471],
         [-0.00588235, -0.01764706,  0.        ,  0.01764706,  0.00588235,
           0.01764706],
         [-0.00588235, -0.01764706,  0.00588235,  0.01764706,  0.00588235,
           0.01764706]]))

# Compare thiery and PyTorch implementation

# Plotting 

In [26]:
def plot_all(integer_space,expression,parameter):
    
    quantized_tensor = np.array(torch.quantize_per_tensor(torch.tensor(integer_space, dtype=torch.float32), scale_symmetric(integer_space), 0, torch.qint8).int_repr())
    
    quantized_integer_space = quantization_operator(integer_space,scale_symmetric(integer_space),0)
    
    quantized_tensor_asymmetric_scheme=np.array(torch.quantize_per_tensor(torch.tensor(integer_space, dtype=torch.float32), scale_asymmetric(integer_space), zero_point_asymmetric(integer_space), torch.qint8).int_repr())
    
    quantized_integer_space_asymmetric_scheme=quantization_operator(integer_space,scale_asymmetric(integer_space),zero_point_asymmetric(integer_space))
    
    # Create the figure
    fig = go.Figure()
    
    # Original values
    fig.add_scatter(
        x=integer_space,
        y=np.zeros(len(integer_space)),
        mode='markers+text',
        name='Original',
        marker=dict(color='blue'),
        text=[f"{v:.4f}" for v in integer_space],
        textposition="top center"
    )
    fig.update_layout(
        xaxis_title="Input Value (Floating Point)",
        yaxis_title="Vertical Offset (for separation)",
        title= f"Experiment with {parameter} data"
    )
    fig.show()
    
    fig = go.Figure()
    fig.add_scatter(
        x=quantized_tensor,
        y=np.zeros(len(quantized_tensor))+0.25,
        mode='markers+text',
        name='Quantized by PyTorch (symmetric scheme)',
        marker=dict(color='red'),
        text=[f"{v:.4f}" for v in quantized_tensor],
        textposition="top center"
    )
    fig.add_scatter(
        x=quantized_integer_space,
        y=np.zeros(len(quantized_integer_space))+0.5,
        mode='markers+text',
        name='Quantized by theory (symmetric scheme)',
        marker=dict(color='green'),
        text=[f"{v:.4f}" for v in quantized_integer_space],
        textposition="top center"
    )
    fig.add_scatter(
        x=quantized_integer_space_asymmetric_scheme,
        y=np.zeros(len(quantized_integer_space_asymmetric_scheme))+0.75,
        mode='markers+text',
        name='Quantized by theory (asymmetric scheme)',
        marker=dict(color='blue'),
        text=[f"{v:.4f}" for v in quantized_integer_space_asymmetric_scheme],
        textposition="top center"
    )
    fig.add_scatter(
        x=quantized_tensor_asymmetric_scheme,
        y=np.zeros(len(quantized_tensor_asymmetric_scheme))+1,
        mode='markers+text',
        name='Quantized by PyTorch (asymmetric scheme)',
        marker=dict(color='purple'),
        text=[f"{v:.4f}" for v in quantized_tensor_asymmetric_scheme],
        textposition="top center"
    )
    fig.update_layout(
        xaxis_title="Input Value (Floating Point)",
        yaxis_title="Vertical Offset (for separation)",
        title= f"Experiment with {parameter} data {expression}"
    )
    fig.show()

In [27]:
integer_space_symmetric = np.linspace(-1, 1, 15, dtype=float)
plot_all(integer_space_symmetric,"np.linspace(-1, 1, 15, dtype=float)","symmetric")

Asymetric data

In [28]:
integer_space_asymmetric_positive = np.linspace(-2, 5, 15, dtype=float)
plot_all(integer_space_asymmetric_positive,"np.linspace(-2, 5, 15, dtype=float)","asymmetric")

In [29]:
integer_space_asymmetric_negative = np.linspace(-5, 3, 15, dtype=float)
plot_all(integer_space_asymmetric_negative,"np.linspace(-5, 3, 15, dtype=float)","asymmetric")

# Normal distribution

In [30]:
mu, sigma = 0, 0.1 # mean and standard deviation
s = np.random.normal(mu, sigma, 1000)

In [31]:
abs(mu - np.mean(s)), abs(sigma - np.std(s, ddof=1))

(np.float64(0.0017843294253523476), np.float64(0.004289913513604415))

In [32]:
original_tensor=s
quantized_tensor = np.array(torch.quantize_per_tensor(torch.tensor(s, dtype=torch.float32), scale_symmetric(s), 0, torch.qint8).int_repr())
quantized_tensor_asym = np.array(torch.quantize_per_tensor(torch.tensor(s, dtype=torch.float32), scale_asymmetric(s), zero_point_asymmetric(s), torch.qint8).int_repr())
quantized_integer_space = quantization_operator(s,scale_symmetric(s),0)
quantized_integer_space_asymmetric_scheme=quantization_operator(s,scale_asymmetric(s),zero_point_asymmetric(s))

In [33]:
fig = go.Figure()
# Original values
fig.add_trace(go.Histogram(x=original_tensor,name='Original tensor'))
fig.update_layout(
    xaxis_title="Input Value (Floating Point)",
    yaxis_title="Vertical Offset (for separation)",
    title= "Experiment with Gaussian Distribution. Results before quantization"
)
fig.show()

fig = go.Figure()
fig.add_trace(go.Histogram(x=quantized_tensor,name='Quantized by PyTorch (symmetric scheme)'))
print("Pytorch symmetric",min(quantized_tensor),max(quantized_tensor))
fig.add_trace(go.Histogram(x=quantized_tensor_asym,name='Quantized by PyTorch (asymmetric scheme)'))
print("Pytorch asymmetric",min(quantized_tensor_asym),max(quantized_tensor_asym))
fig.add_trace(go.Histogram(x=quantized_integer_space,name='Quantized by theory (symmetric scheme)'))
print("Theory symmetric",min(quantized_integer_space),max(quantized_integer_space))
fig.add_trace(go.Histogram(x=quantized_integer_space_asymmetric_scheme,name='Quantized by theory (asymmetric scheme)'))
print("Theory asymmetric",min(quantized_integer_space_asymmetric_scheme),max(quantized_integer_space_asymmetric_scheme))

fig.update_layout(
    xaxis_title="Input Value (Floating Point)",
    yaxis_title="Vertical Offset (for separation)",
    title= "Experiment with Gaussian Distribution. Results after quantization"
)
fig.show()

Pytorch symmetric -113 127
Pytorch asymmetric -112 127
Theory symmetric -113.0 127.0
Theory asymmetric -128.0 126.0


# Difference between INT8 and QINT8

In [34]:
input_data=np.linspace(-2, 5, 15, dtype=float)
tensor= torch.quantize_per_tensor(torch.tensor(input_data, dtype=torch.float32), scale_symmetric(input_data), 0, torch.qint8)
quantized_tensor = np.array(tensor.int_repr())
quantized_integer_space = quantization_operator(input_data,scale_symmetric(input_data),0)
print("input data",input_data)
print("dtype.qint8 with .int_repr()",quantized_tensor)
print("dtype.qint8 as tensor",tensor)
print("dtype.int8 after appling the theory",quantized_integer_space)

input data [-2.  -1.5 -1.  -0.5  0.   0.5  1.   1.5  2.   2.5  3.   3.5  4.   4.5
  5. ]
dtype.qint8 with .int_repr() [-51 -38 -26 -13   0  13  26  38  51  64  76  89 102 115 127]
dtype.qint8 as tensor tensor([-2.0000, -1.4902, -1.0196, -0.5098,  0.0000,  0.5098,  1.0196,  1.4902,
         2.0000,  2.5098,  2.9804,  3.4902,  4.0000,  4.5098,  4.9804],
       size=(15,), dtype=torch.qint8,
       quantization_scheme=torch.per_tensor_affine, scale=0.0392156862745098,
       zero_point=0)
dtype.int8 after appling the theory [-51. -38. -25. -12.   0.  12.  25.  38.  51.  63.  76.  89. 102. 114.
 127.]
